# 08_01 Loading GPT-2's weights by hand

*📖 Book: section 5.5 (loading pretrained weights from OpenAI)*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → model learns → model writes → MCP tool
> ```
>
> You are here: **after the main course**: getting the adult who already speaks English
>
> Project files this connects to:
>
> - `src/homer_ai/gpt.py`
> - `src/homer_ai/pretrained.py`
>
> **This notebook:** OpenAI's GPT-2 is one file of 548 MB. What's inside it, and how do its numbers get into the `GPTModel` we wrote in chapter 4?
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- What a weights file really is: a list of names, shapes and raw numbers
- How to read a safetensors file with plain Python, no extra library
- How GPT-2's names and table layouts map onto our `GPTModel`
- How to check the loading worked: the model writes English

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **safetensors** | a simple file format for weights: a short text header that lists every tensor, then all the numbers as raw bytes | `model.safetensors`, 548 MB |
| **header** | the first part of a file, which describes the rest | `"wpe.weight": {"dtype": "F32", "shape": [1024, 768], …}` |
| **float32** | a number with a decimal point, stored in 4 bytes; the usual type of a weight | 1024 × 768 float32 numbers take 3,145,728 bytes |
| **SHA-256** | a fingerprint of a file: 64 characters that change completely if one byte of the file changes | `248dfc39…a707` for GPT-2's file |
| **weight tying** | using one table for two jobs: GPT-2's token table is also its output head | `wte` is used at the bottom and at the top |

## 3. The idea

### ❓ What does it mean to "load GPT-2"?

A trained model is two things: the code that does the maths, and the numbers the maths uses (the weights). The code we already have, because `src/homer_ai/gpt.py` is built like GPT-2. So all we need is OpenAI's numbers.

Think of our `GPTModel` as an empty form, and of GPT-2's file as a filled-in copy of nearly the same form, from another office. The boxes are the same, but they're named differently, and a few are laid out the other way round. This notebook copies each box across.

```
model.safetensors (548 MB)          our GPTModel (gpt.py)
  wte.weight        ───────────▶    token_embedding.weight
  wpe.weight        ───────────▶    position_embedding.weight
  h.0.ln_1.weight   ───────────▶    blocks.0.norm1.weight
  …                                 …
```

In the picture of [08_00](08_00_chapter_overview.ipynb), this is where we get the adult who already speaks English.

### ❓ Why by hand? Isn't there a library for this?

There is. With the `transformers` library it's one line. But then you'd have a model you can't see inside, and the point of this course is that there's no magic in it. The file format is simple enough to read with the Python standard library, so we do that, and add no new dependency.

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ Step 1: where do the weights come from?

From Hugging Face, a site where people share models. The file is `model.safetensors` in the `openai-community/gpt2` repository. `download_gpt2` (in `pretrained.py`) fetches it once, to `~/.cache/homer-ai/gpt2/`, and shows progress. After that it only checks that the file is there.

A download can break or be tampered with, so we check two things: the size, and the **SHA-256**, a fingerprint of the whole file. The helper code lives in `src/homer_ai/`: `pretrained.py`, `finetune.py` and `lora.py`, imported like the rest of the project.

In [ ]:
from homer_ai.pretrained import GPT2_BYTES, GPT2_SHA256, download_gpt2, sha256_of

path = download_gpt2()  # 548 MB the first time; after that, nothing to do
print(path)
print(f"size:    {path.stat().st_size:,} bytes  (expected {GPT2_BYTES:,})")
print("SHA-256:", sha256_of(path))
print("matches: ", sha256_of(path) == GPT2_SHA256)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

548,105,171 bytes, and the fingerprint `248dfc39…a707` matches the one written in `pretrained.py`. So this is byte for byte the file we expect.

</details>

### ❓ Step 2: what's inside the file?

Three parts, one after the other:

```
┌─────────┬──────────────────────────────┬──────────────────────────────────┐
│ 8 bytes │ N bytes                      │ the rest                         │
│ N       │ the header: a JSON text that │ the numbers of every tensor,     │
│         │ lists every tensor           │ as raw bytes, back to back       │
└─────────┴──────────────────────────────┴──────────────────────────────────┘
```

The first 8 bytes are one whole number: how long the header is. `struct.unpack("<Q", …)` turns 8 bytes into that number (`<` means the smallest byte comes first, which is called little-endian; `Q` means an 8-byte whole number without a sign).

👉 **Predict first:** the file is 548 MB. Will the header be a big part of it, or tiny?

In [ ]:
import json
import struct

with open(path, "rb") as f:  # "rb": read the raw bytes, not text
    (header_length,) = struct.unpack("<Q", f.read(8))
    header = json.loads(f.read(header_length))

print("header length:", f"{header_length:,} bytes")
print("entries:      ", len(header))
print()
print('"wpe.weight":', header["wpe.weight"])

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The header is 14,283 bytes, a tiny part of 548 MB. It has 161 entries: one per tensor (160 of them), and one small note called `__metadata__` that we skip. The entry for `wpe.weight` (GPT-2's position table) says: the numbers are `F32` (float32, 4 bytes each), the shape is 1024 × 768, and they sit at bytes 267,533,312 to 270,679,040 of the data part.

Check it: 1024 × 768 × 4 = 3,145,728, and 270,679,040 − 267,533,312 = 3,145,728. It fits.

</details>

### ❓ Step 3: how do I get one tensor out?

Jump to where its bytes start, read them, and tell PyTorch what they are. The data part begins right after the header, at byte `8 + header_length`.

`torch.frombuffer` takes raw bytes and reads them as numbers of the type you name. It gives one long row, so we `reshape` it to the shape the header promised.

In [ ]:
import torch

entry = header["wpe.weight"]
begin, end = entry["data_offsets"]
with open(path, "rb") as f:
    f.seek(8 + header_length + begin)  # jump to the first byte of this tensor
    raw = bytearray(f.read(end - begin))

wpe = torch.frombuffer(raw, dtype=torch.float32).reshape(entry["shape"])
print("bytes read:", f"{len(raw):,}")
print("shape:     ", tuple(wpe.shape))
print("position 0, first 4 numbers:", [round(v, 4) for v in wpe[0, :4].tolist()])

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

3,145,728 bytes became a table of shape `(1024, 768)`: one row of 768 numbers for each of the 1,024 positions GPT-2 can read. Row 0 starts `-0.0188, -0.1974, 0.004, 0.0113`. Somebody's training run found those numbers in 2019, and now they're in your notebook.

`read_safetensors` in `pretrained.py` is this cell in a loop over all 160 entries.

</details>

### ❓ Step 4: which tensors are there, and do they match ours?

Let's read them all and look at the names. GPT-2 calls its blocks `h.0` to `h.11`. We print block 0, and the few tensors outside the blocks.

👉 **Predict first:** our attention has three layers, `query`, `key` and `value`. Do you expect to find three in GPT-2's list?

In [ ]:
from homer_ai.pretrained import read_safetensors

theirs = read_safetensors(path)  # a dict: name -> tensor
for name in sorted(theirs):
    if name.startswith("h.0.") or not name.startswith("h."):
        print(f"{name:26} {tuple(theirs[name].shape)}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

There's no `query`, `key` or `value`. There's one layer, `h.0.attn.c_attn`, with a weight of shape `(768, 2304)`. And 2304 = 3 × 768. GPT-2 keeps the three side by side in **one** table. That's the first difference.

The rest is the parts we know from chapter 4, under shorter names: `wte` (the token table, 50,257 × 768), `wpe` (the position table), `ln_1` and `ln_2` (the two layer norms of a block), `attn.c_proj` (attention's output projection), `mlp.c_fc` and `mlp.c_proj` (the two layers of the feed-forward step), and `ln_f` (the final layer norm).

`h.0.attn.bias` is not a bias at all. It's the causal mask of [03_03](../03_attention/03_03_hiding_the_future.ipynb), saved as a table of ones and zeros. Our model builds its own mask, so we skip it.

</details>

Here's the whole mapping in one picture. The orange row is the one that needs cutting.

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import (
    ACCENT,
    EDGE,
    GOOD,
    INK,
    MUTED,
    arrow,
    box,
    diagram_axes,
    studio_figure,
    tag,
)

fig = studio_figure("GPT-2's weights", "The same numbers, under our names", (10, 8.6))
ax = diagram_axes(fig)
tag(ax, 2.3, 7.1, "in GPT-2's file", INK, 11, weight="bold")
tag(ax, 7.9, 7.1, "in our GPTModel (gpt.py)", INK, 11, weight="bold")

# (GPT-2's name, its height, [(our name, its height)], colour)
rows = [
    ("wte", 6.5, [("token_embedding", 6.5), ("output_head  (a copy)", 5.85)], GOOD),
    ("wpe", 5.2, [("position_embedding", 5.2)], EDGE),
    ("h.N.ln_1", 4.55, [("blocks.N.norm1", 4.55)], EDGE),
    (
        "h.N.attn.c_attn",
        3.25,
        [
            ("…attention.query", 3.9),
            ("…attention.key", 3.25),
            ("…attention.value", 2.6),
        ],
        ACCENT,
    ),
    ("h.N.attn.c_proj", 1.95, [("…attention.out_proj", 1.95)], EDGE),
    ("h.N.ln_2", 1.3, [("blocks.N.norm2", 1.3)], EDGE),
    ("h.N.mlp.c_fc, c_proj", 0.65, [("…feed_forward.0, .2", 0.65)], EDGE),
    ("ln_f", 0.0, [("final_norm", 0.0)], EDGE),
]
for gpt2_name, y, our_names, color in rows:
    box(ax, 2.3, y, 3.3, 0.48, gpt2_name, color, mono=True, fontsize=10)
    for our_name, our_y in our_names:
        box(ax, 7.9, our_y, 3.5, 0.48, our_name, color, mono=True, fontsize=10)
        line = MUTED if color == EDGE else color
        arrow(ax, (3.95, y), (6.15, our_y), color=line)
tag(ax, 5.05, 3.75, "one table,", ACCENT, 9)
tag(ax, 5.05, 2.72, "cut in three", ACCENT, 9)
ax.plot([0.35, 0.35], [0.45, 4.75], color=MUTED, linewidth=2, zorder=3)
ax.text(0.12, 2.6, "N = 0 … 11: the 12 blocks", color=MUTED, fontsize=9,
        rotation=90, ha="center", va="center")
ax.set_ylim(-0.45, 7.45)
plt.show()

### ❓ Step 5: how do we cut the one table into query, key and value?

First a detail about layout. Our `nn.Linear` stores its weight as `(outputs, inputs)`. GPT-2's layers store it as `(inputs, outputs)`, the other way round. So every GPT-2 weight table has to be turned on its side with `.T` (transpose: rows become columns).

For `c_attn` that gives `(2304, 768)`: 2,304 output rows. The first 768 rows are the query layer, the next 768 the key layer, the last 768 the value layer.

```
c_attn.weight  (768, 2304)   ── .T ──▶  (2304, 768)
                                         rows    0 …  767   → query.weight  (768, 768)
                                         rows  768 … 1535   → key.weight    (768, 768)
                                         rows 1536 … 2303   → value.weight  (768, 768)
```

How do we know we cut it right? One fused layer and three separate layers must give the same numbers for the same input. Let's check on a random row of 768 numbers.

In [ ]:
from torch import nn

fused_w = theirs["h.0.attn.c_attn.weight"].T  # (2304, 768)
fused_b = theirs["h.0.attn.c_attn.bias"]  # (2304,)

torch.manual_seed(0)
x = torch.randn(768)  # stands for one token on its way into attention
together = fused_w @ x + fused_b  # GPT-2's way: 2304 numbers at once

parts = {}
for i, name in enumerate(["query", "key", "value"]):
    rows = slice(i * 768, (i + 1) * 768)
    layer = nn.Linear(768, 768)  # our way: one layer each
    layer.load_state_dict({"weight": fused_w[rows], "bias": fused_b[rows]})
    parts[name] = layer(x)
    shapes = tuple(layer.weight.shape), tuple(layer.bias.shape)
    print(f"{name:6} weight {shapes[0]}  bias {shapes[1]}")

ours = torch.cat([parts["query"], parts["key"], parts["value"]])
print("same numbers as the fused layer:", torch.allclose(together, ours, atol=1e-5))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Three layers of `(768, 768)`, each with a bias, and together they give the same 2,304 numbers as GPT-2's one layer. So the cut is right.

Notice the bias. Our Homer GPT has no bias in query, key and value (`qkv_bias=False`). GPT-2 has one, so the config for it says `qkv_bias=True`.

</details>

### ❓ Step 6: how does everything get into our model?

`gpt2_to_ours` in `pretrained.py` does steps 4 and 5 for all 12 blocks: it renames every tensor, transposes the weight tables, and cuts `c_attn` in three. The result is a dict with **our** names. Then PyTorch's `load_state_dict` copies it into the model, and complains if a name is unknown or a shape is wrong. That check is what makes me trust the mapping.

One box is left over: our `output_head`. GPT-2 has no tensor for it. It reuses the token table `wte` for the output (that's called **weight tying**), so we copy `wte` into our head.

👉 **Predict first:** GPT-2 small is known as a 124-million-weight model. Will our model report 124 million too?

In [ ]:
from homer_ai.gpt import GPTModel
from homer_ai.pretrained import GPT2_CONFIG, gpt2_to_ours

print(GPT2_CONFIG)
model = GPTModel(GPT2_CONFIG)  # our class, GPT-2's sizes, random numbers for now
result = model.load_state_dict(gpt2_to_ours(theirs), strict=False)
model.eval()

masks = [name for name in theirs if name.endswith(".attn.bias")]  # no learned numbers
learned = sum(t.numel() for name, t in theirs.items() if name not in masks)
in_ours = sum(p.numel() for p in model.parameters())
print()
untouched = result.missing_keys
print("left untouched in our model:", len(untouched), "tensors, e.g.", untouched[0])
print("names we don't have:        ", result.unexpected_keys)
print(f"learned numbers in the file: {learned:>12,}")
print(f"weights in our model:        {in_ours:>12,}")
print(f"difference:                  {in_ours - learned:>12,}  = 50,257 x 768")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Every name found its place: the only 12 tensors left untouched are our own causal masks, one per block, and there are no unknown names.

The file has 124,439,808 learned numbers. Our model has 163,037,184. The difference is exactly 50,257 × 768 = 38,597,376: the copy of `wte` in our output head. GPT-2 stores that table once and uses it twice; we store it twice. The numbers are the same, so the model computes the same thing.

</details>

### ❓ Step 7: did it work? Can it write English?

The real test. If one table were in the wrong place or the wrong way round, we'd get rubbish.

GPT-2 doesn't read characters. It reads BPE tokens ([02_03](../02_text_to_numbers/02_03_bpe.ipynb)), and `tiktoken` has its exact vocabulary ([A_05](../A_toolbox/A_05_tiktoken_bpe.ipynb)). `continue_text` is the writing loop of [06_01](../06_making_it_talk/06_01_writing_greek.ipynb): score the next token, pick one, add it, repeat.

We give the same prompt to two models: our class with random numbers, and our class with GPT-2's numbers. Greedy writing, so always the most likely token.

👉 **Predict first:** what will the random one write?

In [ ]:
import tiktoken

from homer_ai.pretrained import continue_text

encoding = tiktoken.get_encoding("gpt2")
prompt = "Every effort moves you"
print("prompt as tokens:", [encoding.decode([i]) for i in encoding.encode(prompt)])

torch.manual_seed(0)
random_model = GPTModel(GPT2_CONFIG).eval()  # same class, untrained
print()
print("random weights:", repr(continue_text(random_model, encoding, prompt, 12)))
print("GPT-2 weights: ", repr(continue_text(model, encoding, prompt, 20)))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The random model writes token soup: `Every effort moves youandalrarilywestNAT nerds Bearing…`. Same code, no knowledge.

With GPT-2's numbers the same code writes `Every effort moves you forward.\n\nThe first step is to understand the importance of your work.` That's real English, and it's the same continuation other GPT-2 implementations give for this prompt with greedy writing. So the weights are in the right boxes.

</details>

### ❓ What does it make of Homer?

Let's give it Butler's first words, and let chance in with top-k ([06_02](../06_making_it_talk/06_02_less_repetition.ipynb)): one of the 40 most likely tokens each time.

In [ ]:
prompt = "Tell me, O Muse, of"
print("greedy:", repr(continue_text(model, encoding, prompt, 30)))
print("top-40:", repr(continue_text(model, encoding, prompt, 30, top_k=40, seed=1)))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Greedy loops, as greedy does: `…of what you have done to me?\n\nO Muse, of what you have done to me?` With top-40 it writes fluent English that goes its own way, something like `…of what you have done." "Hush, yes. That said, you've done very good.`

So it writes proper sentences, which our Homer model never did. But it isn't Butler and it isn't the Odyssey. It continues in the style of the web pages it learned from. That's the adult who speaks English, with no course for any job yet.

</details>

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ Is it safe to open a weights file from the internet?

With safetensors, yes, and that's why the format exists. It holds only a JSON text and raw numbers, so reading it can't run any code. Older `.pt` or `.bin` files can contain code that runs when you open them, which is why our own checkpoints are loaded with `weights_only=True` ([05_06](../05_training_on_the_odyssey/05_06_saving_and_loading.ipynb)). The SHA-256 check on top tells us the file is the one we meant to get.

### ❓ Is our model now exactly GPT-2?

Very nearly. One small thing differs: between the two feed-forward layers, OpenAI's code uses a quick approximation of the GELU curve ([04_02](../04_building_the_model/04_02_feed_forward.ipynb)), and our `nn.GELU()` uses the exact curve. The two curves differ by less than 0.001 everywhere, and the text above came out the same. For fine-tuning it makes no difference.

### ❓ Why 1,024 positions, when we only gave it a few tokens?

1,024 is the most it can read at once (its context length). Shorter text just uses the first rows of the position table, as in [02_06](../02_text_to_numbers/02_06_positions.ipynb).

### ❓ Can I load the bigger GPT-2 models the same way?

Yes. GPT-2 medium, large and XL use the same names and layout, with more blocks and wider rows. You'd change the URL and the sizes in the config. They're 1.5 GB to 6 GB, and slow on a CPU, so this course stays with small.

## 5. Recap

- A weights file is a header (names, shapes, where the bytes are) plus raw numbers. A safetensors file can be read with `struct`, `json` and `torch.frombuffer`.
- GPT-2's tensors map one to one onto our `GPTModel`. Three things need care: other names, weight tables stored the other way round (`.T`), and query, key and value fused into one table.
- GPT-2 ties its output head to its token table, so our model holds 163 million numbers for GPT-2's 124 million.
- The check is to make it write: the same class gives token soup with random numbers, and English with GPT-2's.
- `load_gpt2()` in `pretrained.py` does all of it in one call. The next notebooks start from there.

**Next:** [08_02: A spam classifier](08_02_spam_classifier.ipynb).